Work is [in progress](https://www.wikidata.org/wiki/Wikidata:WikiProject_Indigenous_peoples_of_North_America/Data_model) to develop a series of data models organizing information about indigenous peoples, governments, and lands. In addition to information content about the models designed for humans, we need a technical encoding of data models that can be used to evaluate conformance of items to the models.

This majority of this work is in creating the schemas themselves. The notebook works through a part of that process using the Shape Expression method that Wikidata uses behind their still developing Entity Schemas ("E" identifiers). It is a potential new class that I may build into my wbmaker Python package at some point. It takes a QID for an item we want to test and an EID for a schema we want to test the item against. It uses PyShEx to evaluate these and yield a result object we can print out or do something else with.

In future, I might tweak this to do some other things like take a SPARQL query input to return and evaluate multiple items and report out on what conforms vs. doesn't conform to the schema.

In [1]:
from pyshex import ShExEvaluator
import requests
import os

class WikidataShExValidator:
    def __init__(self, qid: str, eid: str, user_agent: str = None):
        """
        Initialize the validator with an optional user agent.
        
        Args:
            qid: Wikidata item ID (e.g., 'Q123')
            eid: EntitySchema ID (e.g., 'E502')
            user_agent: Custom user agent string. If None, uses default.
        """
        if user_agent is None:
            user_agent = 'WikidataShExValidator/1.0'
        
        self.headers = {
            "User-Agent": user_agent
        }

        self.qid = qid
        self.eid = eid
        self.shexc = None
        self.rdf = None
        self.results = None
    
    def fetch_entityschema(self):
        """
        Fetch ShExC text for a Wikidata EntitySchema (e.g., 'E502')
        using Special:EntitySchemaText.
        """
        url = f"https://www.wikidata.org/wiki/Special:EntitySchemaText/{self.eid}"
        resp = requests.get(url, headers=self.headers)
        resp.raise_for_status()
        self.shexc = resp.text
        return self

    def fetch_rdf(self):
        """
        Fetch RDF data for the Wikidata item in Turtle format.
        """
        rdf_url = f"https://www.wikidata.org/wiki/Special:EntityData/{self.qid}.ttl"
        resp = requests.get(rdf_url, headers=self.headers)
        resp.raise_for_status()
        self.rdf = resp.text
        return self
    
    def eval_item(self):
        """
        Evaluate a Wikidata item against an EntitySchema.
        Requires that fetch_entityschema() and fetch_rdf() have been called first.
        """
        if self.shexc is None:
            raise ValueError("Schema not fetched. Call fetch_entityschema() first.")
        if self.rdf is None:
            raise ValueError("RDF not fetched. Call fetch_rdf() first.")
            
        self.results = ShExEvaluator(
            rdf=self.rdf,
            schema=self.shexc,
            focus=f"http://www.wikidata.org/entity/{self.qid}"
        ).evaluate()
        return self
    
    def validate(self):
        """
        Convenience method: fetch schema, fetch RDF, and evaluate in one call.
        Returns self to allow chaining or access to results.
        """
        self.fetch_entityschema()
        self.fetch_rdf()
        self.eval_item()
        return self

In [2]:
test_cases = {
    'Cherokee Nation': 'Q14708404',
    'Something Else': 'Q736809'
}

for name, qid in test_cases.items():
    print(f"\n{'='*60}")
    print(f"Testing: {name} ({qid})")
    print('='*60)
    
    v = WikidataShExValidator(
        qid=qid,
        eid="E502",
        user_agent=os.environ['WB_BOT_USER_AGENT']
    ).validate()
    
    for r in v.results:
        print("Conforms:", r.result)
        print("Focus:", r.focus)
        print("Reason:", r.reason)



Testing: Cherokee Nation (Q14708404)
Conforms: True
Focus: http://www.wikidata.org/entity/Q14708404
Reason: 

Testing: Something Else (Q736809)
Conforms: False
Focus: http://www.wikidata.org/entity/Q736809
Reason:   Testing wd:Q736809 against shape FederallyRecognizedTribe
    Testing wds:q736809-E74D0BF3-B12D-46A5-9CF9-5F4E1AB51395 against shape InstanceOfFedTribeStatement
      Node: wd:Q1093829 not in value set:
	 {"values": ["http://www.wikidata.org/entity/Q7840353"], "typ...
  Testing wd:Q736809 against shape FederallyRecognizedTribe
       No matching triples found for predicate p:P31
